# APIs & Structured Outputs

#### Where we are

So far you've talked to an LLM like a human — type a prompt, read the answer.

But in a real application, the LLM sits **inside your backend**, and your code must consume its output **programmatically**.

```
User
 ↓
Backend (your code)
 ↓
LLM API
 ↓
JSON
 ↓
Backend (parse + use it)
 ↓
User
```

This is the phase where your backend experience pays off:

***Calling an LLM is just another external API integration.***

The new challenge is making the model return data your code can **trust and parse** — not free-form prose.

### What is an LLM API?

It's an HTTP endpoint. You send a request, you get a response — exactly like any REST API you've called before.

#### The request

You send the **messages** (the roles from Phase 5) plus some settings:

```
POST /v1/chat/completions

{
  "model": "gpt-4o",
  "messages": [
    { "role": "system", "content": "You are a helpful assistant." },
    { "role": "user",   "content": "What is Django?" }
  ],
  "temperature": 0.2
}
```

#### The response

```
{
  "choices": [
    {
      "message": {
        "role": "assistant",
        "content": "Django is a high-level Python web framework..."
      }
    }
  ],
  "usage": {
    "prompt_tokens": 18,
    "completion_tokens": 42,
    "total_tokens": 60
  }
}
```

Notice two familiar things:

```
1. It's request → response (stateless).
   The API does NOT remember past calls.
   YOU resend the conversation each time.

2. usage = tokens = your bill (Phase 1).
```

#### Stateless = you own the memory

```
Every call:
  send system + full conversation so far
  ↓
  get the next assistant message
  ↓
  append it to YOUR stored conversation
```

The "chat memory" lives in your backend, not in the API.

### The Problem: Free Text Is Hard to Use in Code

Suppose you ask:

```
Extract the name, email, and age from this text:
"Hi, I'm Alex, 29, reach me at alex@mail.com"
```

A normal LLM might reply:

```
Sure! The person's name is Alex, they are 29 years old,
and their email address is alex@mail.com.
```

That's lovely for a human, but your backend now has to **scrape** that sentence. The next call might phrase it differently and your parser breaks.

```
You don't want PROSE.
You want DATA.
```

What you actually need:

```
{
  "name": "Alex",
  "email": "alex@mail.com",
  "age": 29
}
```

The rest of this notebook is three increasingly strict ways to get that:

```
JSON Mode          →  "give me valid JSON"
Structured Outputs →  "give me JSON that matches THIS schema"
Function Calling   →  "give me the arguments to call THIS function"
```

### 1. JSON Mode

You ask the API to return **valid JSON** instead of prose.

```
"response_format": { "type": "json_object" }
```

Plus an instruction in the prompt:

```
Extract name, email, and age.
Respond ONLY with a JSON object.
```

Result:

```
{
  "name": "Alex",
  "email": "alex@mail.com",
  "age": 29
}
```

Now your code can do:

```python
import json
data = json.loads(response_text)
```

#### The catch

```
JSON Mode guarantees the SYNTAX is valid JSON.

It does NOT guarantee the SHAPE.
```

The model might still return:

```
{ "full_name": "Alex", "years": 29 }
```

Valid JSON — wrong keys. Your code expected `name` and `age`.

So JSON mode solves "is it parseable?" but not "is it the right structure?". That's what Structured Outputs fixes.

### 2. Structured Outputs

You give the API a **schema** and it forces the model to match it — correct keys, correct types.

Think of it as a contract, just like a Pydantic model or a DB table definition.

```
Schema:
{
  "name":  string,
  "email": string,
  "age":   integer
}
```

The model is now constrained to return exactly:

```
{
  "name": "Alex",
  "email": "alex@mail.com",
  "age": 29
}
```

#### In Python (Pydantic is the common way)

```python
from pydantic import BaseModel

class Person(BaseModel):
    name: str
    email: str
    age: int

# Many SDKs let you pass the model directly:
person = client.parse(messages=[...], response_format=Person)
print(person.age)   # 29, already typed
```

#### JSON Mode vs Structured Outputs

```
JSON Mode          →  valid JSON, shape not guaranteed
Structured Outputs →  valid JSON, shape GUARANTEED
```

For backend code, prefer **Structured Outputs** whenever the SDK/model supports it. It removes a whole class of "the model returned the wrong keys" bugs.

### 3. Function Calling

Same idea as structured outputs, but the schema describes a **function's arguments**.

You tell the model: "Here are functions you may call." The model doesn't run them — it returns **which function** to call and **with what arguments**, as structured JSON.

#### You describe the function

```
{
  "name": "get_weather",
  "description": "Get the current weather for a city",
  "parameters": {
    "city": "string",
    "unit": "celsius | fahrenheit"
  }
}
```

#### The model responds with a call request

User asks: *"What's the weather in Chennai?"*

```
{
  "tool_call": {
    "name": "get_weather",
    "arguments": { "city": "Chennai", "unit": "celsius" }
  }
}
```

#### Important: the model does NOT execute anything

```
Model gives you:   "call get_weather('Chennai')"

YOUR backend:      actually runs the function.
```

The model just produces a **structured request**. What happens next — running the function and feeding the result back — is the full **Tool Calling** loop, which is Phase 7.

```
Function Calling  =  the model can ASK for a function (structured output)

Tool Calling      =  Function Calling + you EXECUTE it + send the result back
```

### The Backend Integration Pattern

Putting it into a real service looks exactly like wrapping any third-party API:

```
Request comes in
      ↓
Build messages (system + user + context)
      ↓
Call LLM API  (with schema / response_format)
      ↓
Receive JSON
      ↓
Validate against schema (e.g. Pydantic)
      ↓
   valid? ──no──▶ retry / fallback / error
      │yes
      ▼
Use the data like any other object
```

#### Production concerns you already know

```
Timeouts        →  LLM calls can be slow; set sensible limits
Retries         →  transient failures + occasional bad JSON
Rate limits     →  APIs cap requests per minute / tokens per minute
Cost            →  tokens = money; log usage
Validation      →  never trust the payload blindly
Secrets         →  API keys in env vars, not in code
```

#### Streaming (good to know)

Responses can be **streamed** token-by-token (like ChatGPT typing) for better perceived latency. Note: streaming and strict structured outputs don't always combine cleanly — stream for chat UIs, use structured outputs for data you must parse.

### Summary & Goal

```
LLM API            →  just an HTTP request/response; stateless
JSON Mode          →  valid JSON, shape NOT guaranteed
Structured Outputs →  valid JSON that matches your schema
Function Calling   →  structured request to call a function (no execution)
```

#### Goal

***Make the LLM return data your code can trust — then treat it like any other API integration.***

Once an LLM can hand you reliable structured data and *ask* to call functions, you're one step away from letting it actually *use* tools.

That step is **Phase 7 — Tool Calling**.